# Part2 KPF-BERT 순수 베이스라인 재학습

목적: 테스트셋 평가에 사용할 순수 텍스트 베이스라인 모델 학습
- 입력: newsTitle [SEP] newsContent (피처 없음)
- 모델 저장을 확실하게 (Kaggle Output에 남도록)

⚠️ 실행 방법: 반드시 **Save Version → Save & Run All (Commit)** 으로 돌려야
   모델이 Output에 영구 저장됨. 인터랙티브로 돌리면 세션 종료 시 사라짐.

In [1]:
import os
for f in os.listdir('/kaggle/input/'):
    print(f)
    path = f'/kaggle/input/{f}'
    if os.path.isdir(path):
        for ff in os.listdir(path):
            print(f'  {ff}')

datasets
  tlarkdus


In [2]:
!pip install transformers accelerate -q

import torch
print(f'GPU 사용 가능: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 89.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cudf-cu12 26.2.1 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incomp

## 1. 데이터 로드 + 샘플링 (기존과 동일)

In [3]:
import pandas as pd
from sklearn.model_selection import train_test_split

TR_PATH  = '/kaggle/input/datasets/tlarkdus/dataset/TR_preprocessed.csv'
VL_PATH  = '/kaggle/input/datasets/tlarkdus/dataset/VL_preprocessed.csv'
SAVE_DIR = '/kaggle/working'  # Output에 저장됨

tr_df = pd.read_csv(TR_PATH)
vl_df = pd.read_csv(VL_PATH)

# Part2만 사용
train_p2 = tr_df[tr_df['partNum'] == 'P2'].reset_index(drop=True)
vl_p2    = vl_df[vl_df['partNum'] == 'P2'].reset_index(drop=True)

# 87,000건 층화 샘플링 (random_state=42, 기존과 완전히 동일)
tr_p2, _ = train_test_split(train_p2, train_size=87000,
                             stratify=train_p2['clickbaitClass'], random_state=42)
tr_p2 = tr_p2.reset_index(drop=True)

print(f'Train Part2: {len(tr_p2):,}건 / Val Part2: {len(vl_p2):,}건')

# 입력 텍스트 구성 (순수 텍스트, 피처 없음)
tr_p2['input_text'] = tr_p2['newsTitle'].fillna('') + ' [SEP] ' + tr_p2['newsContent'].fillna('')
vl_p2['input_text'] = vl_p2['newsTitle'].fillna('') + ' [SEP] ' + vl_p2['newsContent'].fillna('')

print('\n[라벨 분포]')
print('Train:', tr_p2['clickbaitClass'].value_counts().to_dict())
print('Val:  ', vl_p2['clickbaitClass'].value_counts().to_dict())

Train Part2: 87,000건 / Val Part2: 36,910건

[라벨 분포]
Train: {0: 44175, 1: 42825}
Val:   {0: 18741, 1: 18169}


## 2. Dataset + 학습/평가 함수

In [4]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, classification_report
from tqdm import tqdm

class ClickbaitDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=512):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            'input_ids':      encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'labels':         torch.tensor(self.labels[idx], dtype=torch.long)
        }


def train_epoch(model, dataloader, optimizer, device):
    model.train()
    total_loss = 0
    for batch in tqdm(dataloader, desc='학습 중'):
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['labels'].to(device)
        optimizer.zero_grad()
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(dataloader)


def evaluate(model, dataloader, device):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in tqdm(dataloader, desc='평가 중'):
            input_ids      = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels         = batch['labels']
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            preds   = torch.argmax(outputs.logits, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.numpy())

    acc  = accuracy_score(all_labels, all_preds)
    f1   = f1_score(all_labels, all_preds, average='weighted')
    prec = precision_score(all_labels, all_preds, average='weighted')
    rec  = recall_score(all_labels, all_preds, average='weighted')
    print(f'\n📊 Accuracy {acc:.4f} | F1(weighted) {f1:.4f} | Prec {prec:.4f} | Rec {rec:.4f}')
    print(classification_report(all_labels, all_preds, target_names=['낚시성(0)', '정상(1)']))
    return {'accuracy': acc, 'f1': f1, 'precision': prec, 'recall': rec}

print('✅ 함수 정의 완료')

✅ 함수 정의 완료


## 3. 학습 실행 (3 epoch)

매 epoch마다 모델 저장 → 중간에 끊겨도 안전

In [5]:
MODEL_NAME = 'jinmang2/kpfbert'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model     = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
model.to(device)

tr_dataset = ClickbaitDataset(tr_p2['input_text'].tolist(), tr_p2['clickbaitClass'].tolist(), tokenizer)
vl_dataset = ClickbaitDataset(vl_p2['input_text'].tolist(), vl_p2['clickbaitClass'].tolist(), tokenizer)
tr_loader  = DataLoader(tr_dataset, batch_size=16, shuffle=True)
vl_loader  = DataLoader(vl_dataset, batch_size=16, shuffle=False)

optimizer = AdamW(model.parameters(), lr=1e-5)

best_f1 = 0
for epoch in range(3):
    print(f'\n===== Epoch {epoch+1}/3 =====')
    avg_loss = train_epoch(model, tr_loader, optimizer, device)
    print(f'평균 Loss: {avg_loss:.4f}')
    results = evaluate(model, vl_loader, device)

    # 매 epoch 저장 (중간에 끊겨도 보존)
    epoch_path = os.path.join(SAVE_DIR, f'kpfbert_p2_baseline_epoch{epoch+1}.pt')
    torch.save(model.state_dict(), epoch_path)
    print(f'💾 저장: {epoch_path}')

    # 최고 성능 모델 별도 저장
    if results['f1'] > best_f1:
        best_f1 = results['f1']
        best_path = os.path.join(SAVE_DIR, 'kpfbert_p2_baseline_BEST.pt')
        torch.save(model.state_dict(), best_path)
        print(f'🏆 최고 모델 갱신! F1={best_f1:.4f} → {best_path}')

print(f'\n✅ 학습 완료! 최고 F1(macro): {best_f1:.4f}')

config.json:   0%|          | 0.00/622 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/335 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/454M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

... (10 outputs left out)

학습 중: 100%|██████████| 5438/5438 [2:34:31<00:00,  1.70s/it]


평균 Loss: 0.4275


평가 중: 100%|██████████| 2307/2307 [21:24<00:00,  1.80it/s]



📊 Accuracy 0.7634 | F1(weighted) 0.7634 | Prec 0.7635 | Rec 0.7634
              precision    recall  f1-score   support

      낚시성(0)       0.77      0.76      0.77     18741
       정상(1)       0.76      0.77      0.76     18169

    accuracy                           0.76     36910
   macro avg       0.76      0.76      0.76     36910
weighted avg       0.76      0.76      0.76     36910

💾 저장: /kaggle/working/kpfbert_p2_baseline_epoch3.pt
🏆 최고 모델 갱신! F1=0.7634 → /kaggle/working/kpfbert_p2_baseline_BEST.pt

✅ 학습 완료! 최고 F1(macro): 0.7634


## 4. 저장 확인

In [6]:
print('=== /kaggle/working 저장된 파일 ===')
for f in os.listdir(SAVE_DIR):
    if f.endswith('.pt'):
        size_mb = os.path.getsize(os.path.join(SAVE_DIR, f)) / 1024 / 1024
        print(f'  {f}  ({size_mb:.1f} MB)')

print('\n⚠️ 중요: 이 노트북을 [Save Version → Save & Run All (Commit)] 으로 돌려야')
print('   위 .pt 파일들이 Output에 영구 저장됩니다.')
print('   학습 후 Output 탭에서 다운로드하거나, 새 데이터셋으로 만들어 평가에 사용하세요.')

=== /kaggle/working 저장된 파일 ===
  kpfbert_p2_baseline_epoch3.pt  (435.1 MB)
  kpfbert_p2_baseline_epoch2.pt  (435.1 MB)
  kpfbert_p2_baseline_BEST.pt  (435.1 MB)
  kpfbert_p2_baseline_epoch1.pt  (435.1 MB)

⚠️ 중요: 이 노트북을 [Save Version → Save & Run All (Commit)] 으로 돌려야
   위 .pt 파일들이 Output에 영구 저장됩니다.
   학습 후 Output 탭에서 다운로드하거나, 새 데이터셋으로 만들어 평가에 사용하세요.
